# 1. 환경 설정

In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import math
import numpy as np

In [2]:
def setup_data():
    org_path = "../"
    raw_path = f"{org_path}data/raw/"
    prc_path = f"{org_path}data/processed/"
    seed = 42

    print("변수 세팅 완료")
    return org_path, raw_path, prc_path, seed

# 사용 예시 (반환값 9개)
org_path, raw_path, prc_path, seed = setup_data()

변수 세팅 완료


# 2. 테이블 로드

In [3]:
def load_multiple_csv(base_path: str, filenames: list[str]) -> dict[str, pd.DataFrame]:
    """
    여러 CSV 파일을 한 번에 불러와 DataFrame 딕셔너리로 반환합니다.
    
    Parameters
    ----------
    base_path : str
        기본 경로 (예: "data/raw/")
    filenames : list[str]
        확장자를 제외한 파일명 리스트 (예: ["orders", "products", "users"])

    Returns
    -------
    dict[str, pd.DataFrame]
        각 파일명을 key, DataFrame을 value로 하는 딕셔너리
    """
    base = Path(base_path)
    dataframes = {}

    for name in filenames:
        file_path = base / f"{name}.csv"
        if not file_path.exists():
            print(f"파일 없음: {file_path}")
            continue
        df = pd.read_csv(file_path)
        dataframes[name] = df
        print(f"{name}.csv 로드 완료. shape={df.shape}")

    return dataframes

In [4]:
file_list = [
    "distribution_centers", "events", "inventory_items",
    "order_items", "orders", "products", "users"
]

# CSV 로드
dfs = load_multiple_csv(raw_path, file_list)

distribution_centers.csv 로드 완료. shape=(10, 4)
events.csv 로드 완료. shape=(2431963, 13)
inventory_items.csv 로드 완료. shape=(490705, 12)
order_items.csv 로드 완료. shape=(181759, 11)
orders.csv 로드 완료. shape=(125226, 9)
products.csv 로드 완료. shape=(29120, 9)
users.csv 로드 완료. shape=(100000, 15)


In [5]:
# 변수 자동 생성
for name, df in dfs.items():
    globals()[name] = df
    print(f"변수 생성 완료: {name} (shape={df.shape})")

변수 생성 완료: distribution_centers (shape=(10, 4))
변수 생성 완료: events (shape=(2431963, 13))
변수 생성 완료: inventory_items (shape=(490705, 12))
변수 생성 완료: order_items (shape=(181759, 11))
변수 생성 완료: orders (shape=(125226, 9))
변수 생성 완료: products (shape=(29120, 9))
변수 생성 완료: users (shape=(100000, 15))


# 3. 공통 테이블 정의

세션 단위로 예측이 필요하므로 세션 단위로 집계 필요
1) 파생 변수 생성 및 가공
1) 이벤트 개수 기반 Feature
2) 시간/행동 패턴 Feature
3) 결과 이벤트 Feature (Label 생성용)


In [6]:
def preprocess_events(events: pd.DataFrame):
    events = events.copy()

    # 1) 기본 dtype 정리
    events = events.convert_dtypes()

    force_dtypes = {
        'id': 'string',
        'user_id': 'string',
        'sequence_number': 'string',
    }
    events = events.astype({k: v for k, v in force_dtypes.items() if k in events.columns})

    # 2) 시간 파생 변수 생성
    events['created_at'] = pd.to_datetime(events['created_at'], errors='coerce')

    derived_time_cols = ['year', 'month', 'day', 'hour', 'minute', 'second']

    events['year'] = events['created_at'].dt.year.astype('Int64')
    events['month'] = events['created_at'].dt.month.astype('Int64')
    events['day'] = events['created_at'].dt.day.astype('Int64')
    events['hour'] = events['created_at'].dt.hour.astype('Int64')
    events['minute'] = events['created_at'].dt.minute.astype('Int64')
    events['second'] = events['created_at'].dt.second.astype('Int64')

    # 시간 컬럼을 문자열로 (범주형처럼 쓰기 위해)
    for c in derived_time_cols:
        events[c] = events[c].astype('Int64').astype('string')

    # 3) 사용자 ID 존재 여부
    events['is_user_id_present'] = events['user_id'].notna()

    # 4) URI 파생 변수
    def extract_first_two_segments(uri):
        segs = str(uri).split('/')
        return f"/{segs[1]}" if len(segs) >= 2 and segs[1] != '' else uri

    events['uri_splt'] = events['uri'].apply(extract_first_two_segments)

    # 5) 유저별 생성 세션 수
    events['unique_session_count'] = (
        events.groupby('user_id')['session_id']
              .transform('nunique')
    )

    # 6) 한 세션별 활동 수
    events['activity_count'] = (
        events.groupby('session_id')['session_id']
              .transform('size')
    )

    # 7) 컬럼 그룹 정리 (cat_cols만 있으면 된다고 보면 심플함)
    key_cols = ['id', 'user_id', 'session_id', 'ip_address']
    date_cols = ['created_at']

    cat_cols = events.select_dtypes(include=['string', 'category']).columns.tolist()
    cat_cols = [c for c in cat_cols if c not in key_cols + date_cols]

    # 파생 시간 + 플래그 + uri 파생 포함
    cat_cols.extend(derived_time_cols + ['is_user_id_present', 'uri_splt'])

    print("범주형 컬럼:", cat_cols)
    print("키 컬럼:", key_cols)
    print("시간 컬럼:", date_cols, derived_time_cols)

    return events, key_cols, date_cols, cat_cols

In [7]:
events, key_cols, date_cols, cat_cols = preprocess_events(events)

범주형 컬럼: ['sequence_number', 'city', 'state', 'postal_code', 'browser', 'traffic_source', 'uri', 'event_type', 'year', 'month', 'day', 'hour', 'minute', 'second', 'year', 'month', 'day', 'hour', 'minute', 'second', 'is_user_id_present', 'uri_splt']
키 컬럼: ['id', 'user_id', 'session_id', 'ip_address']
시간 컬럼: ['created_at'] ['year', 'month', 'day', 'hour', 'minute', 'second']


In [8]:
import pandas as pd

def build_session_dataset(events: pd.DataFrame, only_identified_users: bool = True) -> pd.DataFrame:
    """
    이벤트 단위(events) 데이터를 세션 단위로 집계한 데이터셋 생성.

    포함 내용
    - 세션 단위 feature (세션 내 행동 요약, 시간/채널/환경 정보 등)
    - label_repurchase: 다음 세션 존재 여부 (0/1, 재구입 여부 분류용 라벨)
    - next_session_gap_min/hour: 다음 세션까지 소요 시간 (재구입 시점 회귀용 타깃)
    - product_seq_uri: 세션 내 product URI 시퀀스 (추천 과제용 시퀀스 라벨)
    - session_order: 유저별 세션 순서 (1,2,3,...)
    - session_type: T1 / T2 / Other 구분

    파라미터
    - only_identified_users:
        True  -> user_id가 있는 세션만 사용 (구입 세션만 대상)
        False -> 전체 events 사용
    """

    df = events.copy()

    # 0) user_id 있는 세션만 사용할지 여부 (재구입 예측 대상)
    if only_identified_users:
        df = df[df['user_id'].notna()].copy()

    # created_at 보정
    df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')

    # 1) 세션 단위 집계 (현재 세션 내부 정보만 feature로 사용)
    session_df = (
        df.groupby(['user_id', 'session_id'])
          .agg(
              # 시간
              session_start_time=('created_at', 'min'),
              session_end_time=('created_at', 'max'),

              # 행동량
              session_event_count=('event_type', 'count'),
              session_product_views=('event_type', lambda x: (x == 'product').sum()),
              session_cart_events=('event_type',   lambda x: (x == 'cart').sum()),
              session_cancel_events=('event_type', lambda x: (x == 'cancel').sum()),
              event_type_nunique=('event_type', 'nunique'),

              # URI 기반 요약
              session_unique_uris=('uri', 'nunique'),
              session_unique_uri_splt=('uri_splt', 'nunique'),
              main_uri_splt=('uri_splt', lambda x: x.value_counts().idxmax()
                                           if x.notna().any() else pd.NA),

              # 환경 정보 (대표값만 사용)
              session_browser=('browser', 'first'),
              session_traffic_source=('traffic_source', 'first'),
              session_city=('city', 'first'),
              session_state=('state', 'first'),
              session_postal_code=('postal_code', 'first'),
          )
          .reset_index()
    )

    # 2) 세션 지속시간 (분/시간)
    session_df['session_duration_min'] = (
        (session_df['session_end_time'] - session_df['session_start_time'])
        .dt.total_seconds() / 60
    )
    session_df['session_duration_hour'] = session_df['session_duration_min'] / 60

    # 3) 세션 시작 기준 시간 파생
    session_df['session_start_hour'] = session_df['session_start_time'].dt.hour
    session_df['session_start_dayofweek'] = session_df['session_start_time'].dt.dayofweek
    session_df['session_is_weekend'] = session_df['session_start_dayofweek'].isin([5, 6]).astype(int)

    # 4) cart 비율 (구매 의도 강도)
    session_df['cart_to_view_ratio'] = session_df.apply(
        lambda row: row['session_cart_events'] / row['session_product_views']
        if row['session_product_views'] > 0 else 0,
        axis=1
    )
    
    # 5) 유저별 세션 정렬 (실제 시간 순서 기준)
    session_df = (
        session_df
        .sort_values(['user_id', 'session_start_time', 'session_id'])
        .reset_index(drop=True)
    )

    # 6) 다음 세션 시작/종료 시점
    session_df['next_session_start_time'] = (
        session_df
        .groupby('user_id', sort=False)['session_start_time']
        .shift(-1)
    )
    session_df['next_session_end_time'] = (
        session_df
        .groupby('user_id', sort=False)['session_end_time']
        .shift(-1)
    )

    # 라벨: 다음 세션(=다음 구매)이 있으면 1, 없으면 0
    session_df['label_repurchase'] = session_df['next_session_end_time'].notna().astype(int)

    # 다음 "구매 시점"까지 걸린 시간(분/시간) – 재구입 시점 예측용 타깃
    session_df['next_session_gap_min'] = (
        (session_df['next_session_end_time'] - session_df['session_end_time'])
        .dt.total_seconds() / 60
    )
    session_df['next_session_gap_hour'] = session_df['next_session_gap_min'] / 60

    # 7) 유저별 세션 순서 (1, 2, 3, ...)
    session_df['session_order'] = (
        session_df
        .groupby('user_id', sort=False)
        .cumcount()
        .add(1)
    )

    # 8) 세션 타입: T1, T2, T3, ...
    session_df['session_type'] = 'T' + session_df['session_order'].astype(str)

    return session_df

In [9]:
session_df = build_session_dataset(events)
session_df

,user_id,session_id,session_start_time,session_end_time,session_event_count,session_product_views,session_cart_events,session_cancel_events,event_type_nunique,session_unique_uris,...,session_start_dayofweek,session_is_weekend,cart_to_view_ratio,next_session_start_time,next_session_end_time,label_repurchase,next_session_gap_min,next_session_gap_hour,session_order,session_type
0,1,bccf01cb-6f3b-4ef7-aaff-0ea67e584334,2022-07-18 10:17:52+00:00,2022-07-20 10:32:05+00:00,10,3,3,0,4,4,...,0.0,0,1.0,2022-07-18 10:52:33+00:00,2022-07-20 11:05:38+00:00,1,33.550000,0.559167,1,T1
1,1,dc670e53-0eb4-4da2-a023-8f505d74e961,2022-07-18 10:52:33+00:00,2022-07-20 11:05:38+00:00,10,3,3,0,4,4,...,0.0,0,1.0,2022-07-18 11:18:36+00:00,2022-07-19 11:29:28+00:00,1,-1416.166667,-23.602778,2,T2
2,1,7ed34a21-9559-4d31-a16f-d87e4c22d343,2022-07-18 11:18:36+00:00,2022-07-19 11:29:28+00:00,10,3,3,0,4,4,...,0.0,0,1.0,NaT,NaT,0,NaN,NaN,3,T3
3,100,5a795c3c-e0b6-40b3-a3ec-8dfc66157847,2023-12-09 13:20:07+00:00,2023-12-09 13:24:46+00:00,5,1,1,0,5,5,...,5.0,1,1.0,NaT,NaT,0,NaN,NaN,1,T1
4,1000,4faaee5f-3fa9-4665-9c36-cb073c42f189,2020-04-10 06:52:02+00:00,2020-04-10 06:59:30+00:00,5,1,1,0,5,5,...,4.0,0,1.0,NaT,NaT,0,NaN,NaN,1,T1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181754,99998,f900a0c8-62c5-4168-9ac8-cd0bde779778,2022-12-16 06:26:15+00:00,2022-12-16 06:32:25+00:00,5,1,1,0,5,5,...,4.0,0,1.0,2023-01-28 06:18:17+00:00,2023-01-28 06:22:59+00:00,1,61910.566667,1031.842778,1,T1
181755,99998,98d1eeb2-2d59-4a94-aa42-477bd10f2f67,2023-01-28 06:18:17+00:00,2023-01-28 06:22:59+00:00,5,1,1,0,5,5,...,5.0,1,1.0,2023-07-22 05:04:36+00:00,2023-07-22 05:13:13+00:00,1,251930.233333,4198.837222,2,T2
181756,99998,841f595e-9f7b-4337-8c4c-08b82a11fc6b,2023-07-22 05:04:36+00:00,2023-07-22 05:13:13+00:00,5,1,1,0,5,5,...,5.0,1,1.0,NaT,NaT,0,NaN,NaN,3,T3
181757,99999,448f8d4c-7cf3-45b3-b6a7-77978e5e4c07,2023-12-22 08:51:44+00:00,2023-12-23 08:59:09+00:00,7,2,2,0,4,4,...,4.0,0,1.0,2023-12-22 11:26:32+00:00,2023-12-23 11:33:17+00:00,1,154.133333,2.568889,1,T1


In [10]:
# 수치 안맞네?
session_df['session_type'].value_counts()

session_type
T1     80044
T2     45048
T3     25435
T4     15584
T5      8064
T6      4039
T7      2055
T8       924
T9       363
T10      136
T11       48
T12       15
T13        3
T14        1
Name: count, dtype: int64

In [11]:
session_df['session_id'].nunique()

181759

In [12]:
session_df.nunique()

user_id                     80044
session_id                 181759
session_start_time         177128
session_end_time           177129
session_event_count             4
session_product_views           4
session_cart_events             4
session_cancel_events           1
event_type_nunique              2
session_unique_uris             2
session_unique_uri_splt         2
main_uri_splt                   5
session_browser                 5
session_traffic_source          5
session_city                 7586
session_state                 228
session_postal_code         15074
session_duration_min         6687
session_duration_hour        6687
session_start_hour             23
session_start_dayofweek         7
session_is_weekend              2
cart_to_view_ratio              1
next_session_start_time     99145
next_session_end_time       99137
label_repurchase                2
next_session_gap_min        82018
next_session_gap_hour       82018
session_order                  14
session_type  

In [13]:
session_df.shape

(181759, 30)